# Ajuste ponderado de la capa de CO2 y holdout externo LAB016–LAB018

Prueba reversible de los filtros causales efectivos **M1** y **M2** y de un **reajuste local de M0** con la misma data y los mismos pesos, manteniendo `theta_natural_full` congelado. La calibración excluye LAB004 y LAB006, conserva LAB005/LAB007/LAB008 con peso 1 e incorpora LAB019/LAB020/LAB021 con peso 2 por su mejor trazabilidad de CO2. LAB011–LAB012 seleccionan la estructura y LAB016–LAB018 constituyen el holdout externo. El notebook no escribe CSV ni imágenes. El etanol observado (columna `Etanol` del Excel, % v/v → g/L ×7.8924) entra al
bracket químico observado y E0 se inicializa desde m1, igual que en el holdout de referencia de LAB019–LAB021.
Las alícuotas de Alcolyzer de m12 se diluyeron (LAB020: 27.5→37.5 mL; LAB021: 32.5→37.5 mL): ese etanol se
corrige por lote (×37.5/27.5 y ×37.5/32.5); Brix y densidad se midieron antes de diluir y no se corrigen.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import re
import sys
from dataclasses import replace
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
import numpy as np
import openpyxl
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "fermentation_model").exists())
FM = ROOT / "fermentation_model"
LAB2026 = FM / "laboratory_2026"
DATA_DIR = FM / "data" / "mem2026" / "LAB019-021"
DATA_016_018_DIR = FM / "data" / "mem2026" / "LAB016-018"
REFERENCE_RESULTS_DIR = LAB2026 / "results" / "lab019_021_theta_full_holdout"

for path in (FM, LAB2026, FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from shared import run_new_must_glycerol_estimability_doe as base
from shared import run_secondary_joint_campaign_doe as joint
from shared import run_secondary_v2_model_evaluation as secondary_v2
from laboratory_2026 import run_estimability_historical_by_medium as natural_loader
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2_cross

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9.5})

LABS = ("LAB019", "LAB020", "LAB021")
FERMENTER = {"LAB019": "F1", "LAB020": "F2", "LAB021": "F3"}
MATCHED_ORIGINAL = {"LAB019": "LAB004", "LAB020": "LAB006", "LAB021": "LAB009"}
COLORS = {"LAB019": "#4c72b0", "LAB020": "#55a868", "LAB021": "#c44e52"}
T0 = pd.Timestamp("2026-09-21 17:00:00")
# Hora y dosis reales del segundo pulso: hoja Nutricion, no un valor heredado del ajuste histórico.
nutrition_wb = openpyxl.load_workbook(
    DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx", read_only=True, data_only=True
)
pulse2_rows = [
    row for row in nutrition_wb["Nutricion"].iter_rows(min_row=2, values_only=True)
    if row[0] in LABS and row[2] == "Pulso 2"
]
nutrition_wb.close()
assert len(pulse2_rows) == len(LABS)
assert {row[0] for row in pulse2_rows} == set(LABS)
PULSE2_DT = {row[0]: pd.Timestamp(row[3]) for row in pulse2_rows}
PULSE2_H = {lab: (stamp - T0).total_seconds() / 3600.0 for lab, stamp in PULSE2_DT.items()}
pulse2_doses = {row[0]: float(row[4]) / 1000.0 for row in pulse2_rows}  # mg/L -> kg/m³
assert len(set(pulse2_doses.values())) == 1, "el modelo usa una dosis común para los tres lotes"
DELTA_N_KG_M3 = next(iter(pulse2_doses.values()))
GRID_H = 0.25
CO2_FACTOR_G_L_H_PER_SCCM = 44.0095 * 60.0 * 0.74 / (1000.0 * 24.16 * 2.0)
assert np.isclose(CO2_FACTOR_G_L_H_PER_SCCM, 0.0404391928807947)
print("ROOT:", ROOT)
print("t0:", T0, "| pulse 2 [h]:", PULSE2_H)

## Parámetros congelados y artefactos protegidos

In [ ]:
THETA_FULL_PATH = LAB2026 / "results" / "estimability_historical_natural" / "theta_natural_full.csv"
THETA_AUX_PATH = LAB2026 / "results" / "final_operational_doe_v2" / "theta_final_operational_v2.csv"
CO2_PARAM_PATH = LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "three_state_co2_parameters.csv"

theta_table = pd.read_csv(THETA_FULL_PATH)
theta_full = dict(zip(theta_table["parameter"], theta_table["theta"].astype(float)))
assert set(base.FULL17) == set(theta_full), (set(base.FULL17) - set(theta_full), set(theta_full) - set(base.FULL17))

theta_aux = pd.read_csv(THETA_AUX_PATH, index_col=0).iloc[:, 0].astype(float).to_dict()
theta_model = {**theta_aux, **theta_full}  # theta_full siempre prevalece en la capa cinética

co2_parameter_table = pd.read_csv(CO2_PARAM_PATH)
co2_parameter_table = co2_parameter_table[
    co2_parameter_table["state"].eq("C_full_theta_refit")
    & co2_parameter_table["calibration_matrix"].eq("natural")
].copy()
co2_params = dict(zip(co2_parameter_table["parameter"], co2_parameter_table["estimate"].astype(float)))
assert set(co2_cross.PARAMETER_BOUNDS).issubset(co2_params)



display(theta_table)
display(co2_parameter_table[["parameter", "estimate", "active_bound"]])


In [ ]:
import hashlib

PROTECTED_PATHS = [
    LAB2026 / "notebooks" / "holdouts" / "lab019_021_theta_full_holdout.ipynb",
    LAB2026 / "notebooks" / "holdouts" / "lab019_021_theta_full_holdout.executed.ipynb",
    THETA_FULL_PATH,
    CO2_PARAM_PATH,
    REFERENCE_RESULTS_DIR / "co2_metrics.csv",
    REFERENCE_RESULTS_DIR / "co2_prediction_rows.csv",
    REFERENCE_RESULTS_DIR / "protocol_lock.json",
    DATA_016_018_DIR / "LAB016-018-offline-measurements.xlsx",
    LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "external_kinetic_holdout_initial_conditions_LAB013_015.csv",
    LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "external_holdout_prediction_rows_A_vs_B.csv",
    LAB2026 / "results" / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010" / "external_holdout_support_LAB016_018.csv",
    LAB2026 / "results" / "lot2_data_preview" / "figures" / "lot2_F1_process_preview.png",
    LAB2026 / "results" / "lot2_data_preview" / "figures" / "lot2_F2_process_preview.png",
    LAB2026 / "results" / "lot2_data_preview" / "figures" / "lot2_F3_process_preview.png",
]

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

protected_before = {str(path): sha256(path) for path in PROTECTED_PATHS}
display(pd.DataFrame({"archivo_protegido": list(protected_before), "sha256_antes": list(protected_before.values())}))

## Datos de la campaña LAB019–LAB021

In [ ]:
def read_csv_window(path, usecols, start, end, chunksize=500_000):
    frames = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        stamp = pd.to_datetime(chunk["timestamp"], errors="coerce")
        keep = stamp.between(start, end)
        if keep.any():
            part = chunk.loc[keep].copy()
            part["timestamp"] = stamp.loc[keep]
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=usecols)


window_start, window_end = T0 - pd.Timedelta(hours=6), T0 + pd.Timedelta(hours=170)
temperature_raw = read_csv_window(
    DATA_DIR / "backup_global_temperatura.csv", ["timestamp", "fermentador", "T", "SP"], window_start, window_end
)
for col in ("T", "SP"):
    temperature_raw[col] = pd.to_numeric(temperature_raw[col], errors="coerce")
co2_raw = read_csv_window(
    DATA_DIR / "backup_global_co2.csv", ["timestamp", "fermentador", "flow_sccm", "flow_filt_sccm", "status"], window_start, window_end
)
for col in ("flow_sccm", "flow_filt_sccm"):
    co2_raw[col] = pd.to_numeric(co2_raw[col], errors="coerce")

TEMP_RESAMPLE = "1min"    # mediana por minuto: elimina outliers y uniformiza el cadence
TEMP_MEDIAN = "20min"     # mediana móvil: suprime ráfagas y spikes del control on/off
TEMP_MEAN = "45min"       # media móvil: suaviza la oscilación de banda del control

temp_log = {}
for _lab in LABS:
    _f = FERMENTER[_lab]
    frame = (temperature_raw[temperature_raw["fermentador"].eq(_f)]
             .dropna(subset=["T"]).sort_values("timestamp")
             .set_index("timestamp")[["T", "SP"]])
    frame = frame.resample(TEMP_RESAMPLE).median()
    t_med = frame["T"].rolling(TEMP_MEDIAN, center=True, min_periods=1).median()
    temp_log[_f] = pd.DataFrame({
        "T_filt": t_med.rolling(TEMP_MEAN, center=True, min_periods=1).mean(),
        "SP": frame["SP"],
    })

fig, axes = plt.subplots(2, 3, figsize=(15.5, 7.8), sharex=True)
for col, lab in enumerate(LABS):
    f = FERMENTER[lab]
    co2 = co2_raw[co2_raw["fermentador"].eq(f)].sort_values("timestamp")
    t_co2 = (co2["timestamp"] - T0).dt.total_seconds().to_numpy() / 3600.0
    step = max(1, len(co2) // 20000)
    ax = axes[0, col]
    ax.plot(t_co2[::step], co2["flow_filt_sccm"].to_numpy(float)[::step], color="#4c72b0", lw=1.8)
    ax.set_title(f"{lab} ({f})", fontsize=11)
    if col == 0:
        ax.set_ylabel("CO2 filtrado [sccm]", fontsize=10)
    ax = axes[1, col]
    tl = temp_log[f]
    t_temp = (tl.index - T0).total_seconds().to_numpy() / 3600.0
    ax.plot(t_temp, tl["T_filt"].to_numpy(float), color="#c44e52", lw=2.2, label="T filtrada (1 min + 20 min + 45 min)")
    ax.plot(t_temp, tl["SP"].to_numpy(float), color="0.55", lw=1.2, ls="--", label="SP del log")
    if col == 0:
        ax.set_ylabel("Temperatura [°C]", fontsize=10)
        ax.legend(fontsize=8, loc="lower right")
    ax.set_xlabel("t desde inoculación [h]", fontsize=10)
    for row in (0, 1):
        axes[row, col].grid(alpha=0.25)
        axes[row, col].set_xlim(-6.0, 170.0)
fig.suptitle("Logger de campaña — CO2 filtrado (arriba) y temperatura filtrada (abajo)", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.97))
display(fig)
plt.close(fig)
print("Filtrado de temperatura: mediana", TEMP_RESAMPLE, "-> mediana móvil", TEMP_MEDIAN,
      "-> media móvil", TEMP_MEAN, "(todas centradas, sin desplazamiento de fase).")

In [ ]:
from scipy.interpolate import RegularGridInterpolator

CARBO_TABLE = DATA_DIR.parent / "Tabla_CO2_disuelto_carbodoseur_long.csv"
carbo_tabla = pd.read_csv(CARBO_TABLE)
piv = carbo_tabla.pivot_table(index="volumen_residual_mL", columns="temperatura_C", values="CO2_mg_L").sort_index()
vol_axis = piv.index.to_numpy(dtype=float)
temp_axis = piv.columns.to_numpy(dtype=float)
assert np.allclose(np.diff(vol_axis), 1.0) and np.allclose(np.diff(temp_axis), 1.0), "grilla de tabla no regular"
carbo_lookup = RegularGridInterpolator((vol_axis, temp_axis), piv.to_numpy(dtype=float),
                                       method="linear", bounds_error=False, fill_value=np.nan)

offline = pd.read_excel(DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx", sheet_name="MedicionesOffline")
offline["muestra_num"] = pd.to_numeric(offline["Muestra"], errors="coerce")
offline = offline[offline["muestra_num"].notna()].copy()          # PI (pre-inóculo) queda fuera
offline["dt"] = pd.to_datetime(offline["Fecha"].astype(str) + " " + offline["Hora"].astype(str), errors="coerce")
n_sin_fecha = int(offline["dt"].isna().sum())
offline = offline.dropna(subset=["dt"])                            # descarta únicamente filas sin fecha/hora válida
offline["t_h"] = (offline["dt"] - T0).dt.total_seconds() / 3600.0

dissolved_rows = []
for r in offline.itertuples():
    tl = temp_log[r.Fermentador]
    t_arr = (tl.index - T0).total_seconds().to_numpy() / 3600.0
    t_log_c = float(np.interp(r.t_h, t_arr, tl["T_filt"].to_numpy(dtype=float)))
    t_used = float(np.clip(t_log_c, temp_axis[0], temp_axis[-1]))
    lectura = float(r.Vol_Residual_mL) if pd.notna(r.Vol_Residual_mL) else np.nan
    co2_mg_l = float(carbo_lookup(np.array([[lectura, t_used]]))[0]) if np.isfinite(lectura) else np.nan
    dissolved_rows.append({
        "lab": r.CodigoLab, "muestra": int(r.muestra_num), "t_h": float(r.t_h),
        "lectura_carbodoseur_mL": lectura,
        "T_anton_paar_C_ref": float(r.Temperatura_C) if pd.notna(r.Temperatura_C) else np.nan,
        "T_log_filtrada_C": t_log_c,
        "T_recortada_a_techo_tabla": bool(t_used != t_log_c),
        "CO2_disuelto_g_L": co2_mg_l / 1000.0 if np.isfinite(co2_mg_l) else np.nan,
    })
dissolved = pd.DataFrame(dissolved_rows)
print(f"Filas sin Fecha/Hora descartadas: {n_sin_fecha}; lecturas Carbodoseur: {dissolved['CO2_disuelto_g_L'].notna().sum()}.")
display(dissolved.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4), sharex=True)
for ax, lab in zip(axes, LABS):
    sub = dissolved[dissolved["lab"].eq(lab)].sort_values("t_h")
    ok = sub["CO2_disuelto_g_L"].notna()
    ax.plot(sub.loc[ok, "t_h"], sub.loc[ok, "CO2_disuelto_g_L"], "o-", color=COLORS[lab], lw=2.2, ms=7,
            label="CO2 disuelto (Carbodoseur, T del log)")
    ax2 = ax.twinx()
    tl = temp_log[FERMENTER[lab]]
    t_arr = (tl.index - T0).total_seconds().to_numpy() / 3600.0
    ax2.plot(t_arr, tl["T_filt"].to_numpy(dtype=float), color="0.6", lw=1.5, alpha=0.85)
    ax2.set_ylabel("T filtrada [°C]", color="0.45", fontsize=8.5)
    ax2.tick_params(axis="y", labelcolor="0.45")
    ax2.grid(False)
    ax.set_title(lab, fontsize=10)
    ax.set_xlabel("t desde inoculación [h]")
    ax.grid(alpha=0.25)
    ax.set_xlim(-3.0, 100.0)
axes[0].set_ylabel("CO2 disuelto [g L$^{-1}$]")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("CO2 disuelto por Carbodoseur — tabla digitalizada con temperatura del log filtrada", y=1.0)
fig.tight_layout(rect=(0, 0, 1, 0.95))
display(fig)
plt.close(fig)
assert len(dissolved) == len(offline)
print(f"CO2 disuelto calculado: {dissolved['CO2_disuelto_g_L'].notna().sum()} lecturas; sin ajustes ni optimizaciones.")

In [ ]:
from shared.new_must_data_loader import ETHANOL_G_L_PER_PERCENT_VV


def numeric(value):
    try:
        out = float(value)
        return out if np.isfinite(out) else np.nan
    except (TypeError, ValueError):
        return np.nan


def load_offline_samples():
    path = DATA_DIR / "Mediciones-Offline-LAB019-021.xlsx"
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = []
    for excel_row, row in enumerate(wb["MedicionesOffline"].iter_rows(min_row=2, values_only=True), start=2):
        fecha, hora, lab, brix, density = row[0], row[1], row[2], row[5], row[6]
        if lab is None and brix is None and density is None:
            continue
        if lab not in LABS or fecha is None or hora is None or row[4] is None:
            raise ValueError(f"MedicionesOffline fila {excel_row}: metadatos de muestra incompletos")
        hh = hora.time() if isinstance(hora, datetime) else hora
        rows.append({
            "lab": str(lab), "sample": str(row[4]).strip(),
            "sample_timestamp": datetime.combine(fecha.date() if hasattr(fecha, "date") else fecha, hh),
            "Brix": numeric(brix), "density": numeric(density),
            "Etanol_pct_vv": numeric(row[10]), "metadata_fallback": False,
        })
    wb.close()
    result = pd.DataFrame(rows)
    assert not result.duplicated(["lab", "sample"]).any(), "muestras offline duplicadas"
    return result

def load_y15():
    rows = []
    paths = sorted(DATA_DIR.glob("EXPAuto*.txt")) + sorted(DATA_DIR.glob("ONLINE*.txt"))
    for path in paths:
        for line in open(path, encoding="utf-8-sig", errors="replace"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 6:
                continue
            raw = fields[3].strip()
            value = float(raw.replace(",", ".")) if re.fullmatch(r"-?\d+(,\d+)?", raw) else np.nan
            rows.append({"sample_id": fields[0], "analyte": fields[1], "value": value, "raw_value": raw, "analysis_timestamp": fields[5], "source_file": path.name})
    long = pd.DataFrame(rows).drop_duplicates(["sample_id", "analyte"], keep="first")
    wide = long.pivot(index="sample_id", columns="analyte", values="value")
    raw_wide = long.pivot(index="sample_id", columns="analyte", values="raw_value")
    return long, wide, raw_wide


offline = load_offline_samples()
# Dilución Alcolyzer m12, distinta por lote (cuaderno de laboratorio): LAB020 27.5 mL de vino + 10 mL de agua;
# LAB021 32.5 mL de vino + 5 mL de agua (ambas a 37.5 mL). Sólo corrige etanol (medido en la mezcla);
# Brix y densidad se midieron antes de diluir.
DILUTION_FACTORS_M12 = {"LAB020": 37.5 / 27.5, "LAB021": 37.5 / 32.5}
for _lab_m12, _factor in DILUTION_FACTORS_M12.items():
    _sel = offline["lab"].eq(_lab_m12) & offline["sample"].eq("12")
    offline.loc[_sel, "Etanol_pct_vv"] = offline.loc[_sel, "Etanol_pct_vv"] * _factor
assert np.isclose(offline.loc[(offline["lab"] == "LAB020") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 7.33 * 15 / 11)
assert np.isclose(offline.loc[(offline["lab"] == "LAB021") & (offline["sample"] == "12"), "Etanol_pct_vv"].iloc[0], 8.66 * 15 / 13)
y15_long, y15, y15_raw = load_y15()

chem_rows = []
for lab in LABS:
    samples = offline[offline["lab"].eq(lab)].copy().set_index("sample")
    numbered_samples = sorted((item for item in samples.index if str(item).isdigit()), key=int)
    assert "1" in numbered_samples, f"{lab}: falta muestra inicial"
    for sample in numbered_samples:
        sid = f"{lab}-{sample}"
        meta = samples.loc[sample]
        stamp = pd.Timestamp(meta["sample_timestamp"])
        sample_t_h = (stamp - T0).total_seconds() / 3600.0
        t_h = 0.0 if sample == "1" else sample_t_h
        get = lambda name: float(y15.loc[sid, name]) if sid in y15.index and name in y15.columns and pd.notna(y15.loc[sid, name]) else np.nan
        get_raw = lambda name: str(y15_raw.loc[sid, name]) if sid in y15_raw.index and name in y15_raw.columns and pd.notna(y15_raw.loc[sid, name]) else ""
        gf_320_raw = get("GLU-FRU-320")
        gf_low_raw = get("GLUCOSE-FRUCTOSE")
        if np.isfinite(gf_320_raw) and gf_320_raw >= 0.0:
            gf, gf_source = gf_320_raw, "GLU-FRU-320"
        elif np.isfinite(gf_low_raw) and gf_low_raw >= 0.0:
            gf, gf_source = gf_low_raw, "GLUCOSE-FRUCTOSE"
        else:
            gf, gf_source = np.nan, ""
        glucose_320_raw = get("GLUCOSE-320")
        glucose_low_raw = get("GLUCOSE")
        glucose_raw = glucose_320_raw if np.isfinite(glucose_320_raw) else glucose_low_raw
        glucose = glucose_raw if np.isfinite(glucose_raw) and glucose_raw >= 0.0 else np.nan
        fructose = gf - glucose if np.isfinite(gf) and np.isfinite(glucose) and gf >= glucose else np.nan
        pan, ammonia_raw = get("PAN"), get("AMMONIA")
        ammonia = max(ammonia_raw, 0.0) if np.isfinite(ammonia_raw) else np.nan
        yan = pan + 0.823 * ammonia if np.isfinite(pan) and np.isfinite(ammonia) else np.nan
        chem_rows.append({
            "lab": lab, "sample": sample, "sample_timestamp": stamp, "sample_t_h": sample_t_h, "t_h": t_h,
            "metadata_fallback": bool(meta["metadata_fallback"]), "density": float(meta["density"]), "Brix": float(meta["Brix"]),
            "GF_g_l": gf, "GF_source": gf_source,
            "GF_320_raw_g_l": gf_320_raw, "GF_low_raw_g_l": gf_low_raw,
            "GF_320_below_zero": bool(np.isfinite(gf_320_raw) and gf_320_raw < 0.0),
            "G_g_l": glucose, "G_raw_g_l": glucose_raw, "F_g_l": fructose,
            "PAN_mg_l": pan, "NH3_raw_mg_l": ammonia_raw, "NH3_censored_mg_l": ammonia,
            "YAN_mg_l": yan, "N_kg_m3": yan / 1000.0 if np.isfinite(yan) else np.nan,
            "Gly_g_l": get("GLYCEROL"), "Pyr_mg_l": get("PYRUVIC ACID"),
            "Etanol_pct_vv": float(meta["Etanol_pct_vv"]),
            "E_g_l": meta["Etanol_pct_vv"] * ETHANOL_G_L_PER_PERCENT_VV if np.isfinite(meta["Etanol_pct_vv"]) else np.nan,
            "AcAld_mg_l": get("ACETALDEHID-1000"), "Acetate_g_l": get("ACETIC ACID"),
            "G_raw_text": get_raw("GLUCOSE-320") or get_raw("GLUCOSE"),
            "GF_raw_text": get_raw("GLU-FRU-320"),
            "GF_low_raw_text": get_raw("GLUCOSE-FRUCTOSE"),
            "NH3_below_zero": bool(np.isfinite(ammonia_raw) and ammonia_raw < 0.0),
        })
chem = pd.DataFrame(chem_rows).sort_values(["lab", "t_h"]).reset_index(drop=True)
chem["is_initial"] = chem["sample"].eq("1")
chem["soft_flag_GF"] = chem["lab"].eq("LAB019") & chem["sample"].eq("4")
chem["soft_flag_metadata"] = chem["metadata_fallback"]
chem["y15_available"] = chem[["GF_g_l", "YAN_mg_l", "Gly_g_l", "Pyr_mg_l", "AcAld_mg_l"]].notna().any(axis=1)
assert chem[chem["sample"].eq("1")]["E_g_l"].notna().all(), "falta Etanol m1 (inicializa E0)"

assert np.allclose(chem[chem["sample"].eq("1")]["t_h"], 0.0)
assert chem.query("lab == 'LAB020' and sample == '4'")["G_g_l"].isna().all()
assert chem.query("lab == 'LAB020' and sample == '4'")["G_raw_g_l"].iloc[0] == -7.28
assert chem[chem["sample"].eq("12")]["y15_available"].all(), "falta exportación Y15 m12"
assert chem.loc[chem["sample"].eq("12"), "GF_320_below_zero"].all(), "revisar QA de GF m12"
display(chem[["lab", "sample", "sample_t_h", "t_h", "GF_g_l", "G_g_l", "F_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "y15_available", "NH3_below_zero", "metadata_fallback"]])
print("Muestras sin resultado Y15:", chem.loc[~chem["y15_available"], ["lab", "sample"]].to_dict("records"))

In [ ]:
historical_data = natural_loader.load_historical_medium_data("natural")
historical_batches = {b.batch: b for b in joint.make_secondary_batches(historical_data)}
historical_secondary_initials = joint.secondary_initials_by_medium(historical_data)["natural"]

def temperature_profile(lab, grid):
    f = FERMENTER[lab]
    frame = temperature_raw[temperature_raw["fermentador"].eq(f)].dropna(subset=["T"]).sort_values("timestamp")
    t = (frame["timestamp"] - T0).dt.total_seconds().to_numpy() / 3600.0
    return np.interp(grid, t, frame["T"].to_numpy(dtype=float))


holdout_batches = {}
core_predictions = {}
secondary_predictions = {}
initial_rows = []

# Biomasa inicial desde el m1 de Oculyze de cada réplica (~25 min tras la aplicación de
# levadura), con la misma conversión del pipeline histórico (30 pg/célula). La convención
# primaria es Concentración cruda x 0.03, idéntica a la que ancló las X de los originales
# (p.ej. LAB009 t=0: 12.04 x 0.03 = 0.361); la variante con CERO restado se reporta como referencia.
from shared.new_must_data_loader import MILLION_CELLS_ML_TO_KG_M3
assert abs(MILLION_CELLS_ML_TO_KG_M3 - 0.03) < 1e-12

def oculyze_m1_biomass(lab):
    with open(DATA_DIR / f"Oculyze_report_{lab}.csv", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    m1 = next(r for r in rows if r["Name"].strip().upper() == f"{lab}-1")
    cero = next((r for r in rows if r["Name"].strip().upper() == "CERO"), None)
    concentration = float(m1["Concentration"])
    viability = float(m1["Viability"])
    return {
        "x0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3,
        "xd0_kg_m3": concentration * MILLION_CELLS_ML_TO_KG_M3 * (1.0 - viability / 100.0),
        "oculyze_m1_concentration_mcells_ml": concentration,
        "oculyze_m1_viability_pct": viability,
        "x0_cero_subtracted_kg_m3": (
            max(concentration - float(cero["Concentration"]), 0.0) * MILLION_CELLS_ML_TO_KG_M3
            if cero is not None else float("nan")
        ),
    }

oculyze_biomass = {lab: oculyze_m1_biomass(lab) for lab in LABS}
for lab in LABS:
    c = chem[chem["lab"].eq(lab)].sort_values("t_h")
    init = c[c["sample"].eq("1")].iloc[0]
    original = historical_batches[MATCHED_ORIGINAL[lab]]
    f = FERMENTER[lab]
    co2_lab = co2_raw[co2_raw["fermentador"].eq(f)]
    logger_horizon = min(170.0, float((co2_lab["timestamp"].max() - T0).total_seconds() / 3600.0))
    measured_states = ["GF_g_l", "YAN_mg_l", "Gly_g_l", "E_g_l", "Pyr_mg_l", "AcAld_mg_l", "Acetate_g_l"]
    scored_chemistry = c[c[measured_states].notna().any(axis=1)]
    chemistry_horizon = float(scored_chemistry["t_h"].max())
    horizon = max(logger_horizon, chemistry_horizon)
    temp_lab = temperature_raw[temperature_raw["fermentador"].eq(f)].dropna(subset=["T"])
    temperature_last_h = float((temp_lab["timestamp"].max() - T0).total_seconds() / 3600.0)
    grid = np.arange(0.0, horizon + GRID_H / 2.0, GRID_H)
    grid = np.unique(np.r_[grid, c["t_h"].to_numpy(dtype=float), PULSE2_H[lab]])
    grid = grid[(grid >= 0.0) & (grid <= horizon + 1e-9)]
    temp = temperature_profile(lab, grid)
    # E0 desde el Etanol de m1 (% v/v -> g/L); los originales también inicializan E con su ETANOL t=0.
    initials = {
        "X": float(oculyze_biomass[lab]["x0_kg_m3"]), "Xd": float(oculyze_biomass[lab]["xd0_kg_m3"]),
        "N": float(init["N_kg_m3"]), "G": float(init["G_g_l"]), "F": float(init["F_g_l"]),
        "E": float(init["E_g_l"]), "Gly": float(init["Gly_g_l"]),
        "Pyr": float(init["Pyr_mg_l"]),
        "AcAld": float(historical_secondary_initials["AcAld"]),
        "Acetate": float(init["Acetate_g_l"]),
        "O2": float(historical_secondary_initials["O2"]), "CO2": 0.0,
    }
    observations = {state: np.full(len(grid), np.nan) for state in base.STATE_NAMES}
    observations.update({state: np.full(len(grid), np.nan) for state in ("Pyr", "AcAld", "Acetate", "O2")})
    # Química observada sólo se almacena para el diagnóstico condicionado; no altera la simulación cinética.
    for row in c.itertuples():
        idx = int(np.argmin(np.abs(grid - float(row.t_h))))
        if abs(grid[idx] - float(row.t_h)) > 1e-7:
            continue
        for state, value in (("G", row.G_g_l), ("F", row.F_g_l), ("N", row.N_kg_m3), ("Gly", row.Gly_g_l), ("E", row.E_g_l),
        ("Pyr", row.Pyr_mg_l), ("AcAld", row.AcAld_mg_l), ("Acetate", row.Acetate_g_l)):
            observations[state][idx] = value
    batch = base.BatchData(
        medium="natural", batch=lab, time=grid, temperature_c=temp,
        pulses={"N": ((PULSE2_H[lab], DELTA_N_KG_M3),), "G": tuple(), "F": tuple(), "E": tuple(), "X": tuple()},
        observations=observations, initials=initials,
    )
    core = base.simulate(batch, theta_full, grid)
    assert core is not None and np.isfinite(core.to_numpy()).all()
    secondary = secondary_v2.integrate_secondary_v2(batch, theta_model, core)
    assert np.isfinite(secondary.to_numpy()).all()
    holdout_batches[lab], core_predictions[lab], secondary_predictions[lab] = batch, core, secondary
    initial_rows.append({
        "lab": lab, "x0_source": f"Oculyze m1 {lab}-1", **oculyze_biomass[lab],
        "x0_heredado_del_original_kg_m3": float(original.initials["X"]),
        "matched_original_reference": MATCHED_ORIGINAL[lab],
        **initials, "horizon_h": horizon,
        "temperature_extrapolated_h": max(0.0, horizon - temperature_last_h),
    })

initial_table = pd.DataFrame(initial_rows)
display(initial_table)
if initial_table["temperature_extrapolated_h"].gt(0).any():
    print("AVISO: temperatura prolongada con el último valor medido hasta la última química; interpretar los puntos posteriores al logger como sensibilidad.")

In [ ]:
co2_hourly_parts, mask_rows = [], []
for lab in LABS:
    f = FERMENTER[lab]
    frame = co2_raw[co2_raw["fermentador"].eq(f)].sort_values("timestamp").copy()
    frame["t_h"] = (frame["timestamp"] - T0).dt.total_seconds() / 3600.0
    baseline = frame[frame["t_h"].between(-2.0, -0.5)]["flow_sccm"].dropna().to_numpy(dtype=float)
    baseline_center = float(np.median(baseline))
    sigma = float(1.4826 * np.median(np.abs(baseline - baseline_center)))
    threshold = max(3.5 * sigma, 0.5)
    search = frame[frame["t_h"].between(-0.5, 0.75)].copy()
    crossing = search[np.abs(search["flow_sccm"] - baseline_center) > threshold]
    if crossing.empty:
        event_start_h, event_raw_end_h = 0.0, 1.0
    else:
        event_start_h, event_raw_end_h = float(crossing["t_h"].min()), float(crossing["t_h"].max())
    mask_end_h = max(0.0, event_raw_end_h) + 1.5

    zero_support = frame[frame["t_h"].between(mask_end_h, 12.0)]["flow_filt_sccm"].dropna()
    zero_sccm = max(0.0, float(zero_support.quantile(0.10))) if len(zero_support) else 0.0
    frame["flow_corrected_sccm"] = np.maximum(frame["flow_filt_sccm"] - zero_sccm, 0.0)
    hourly = frame.set_index("timestamp")[["flow_corrected_sccm", "flow_filt_sccm", "flow_sccm"]].resample("1h").median().dropna(subset=["flow_corrected_sccm"]).reset_index()
    hourly["t_h"] = (hourly["timestamp"] - T0).dt.total_seconds() / 3600.0
    hourly = hourly[hourly["t_h"].between(0.0, 170.0)].copy()
    hourly["lab"] = lab
    hourly["co2_observed_g_l_h"] = hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM
    hourly["co2_valid"] = hourly["t_h"] > mask_end_h
    hourly["left_censored"] = hourly["co2_observed_g_l_h"] < co2_cross.CO2_DETECTION_LIMIT_G_L_H
    co2_hourly_parts.append(hourly)
    mask_rows.append({
        "lab": lab, "fermenter": f, "baseline_raw_sccm": baseline_center, "baseline_sigma_sccm": sigma,
        "artifact_threshold_sccm": threshold, "event_start_h": event_start_h,
        "event_raw_end_h": event_raw_end_h, "recovery_tail_min": 90, "mask_end_h": mask_end_h,
        "sensor_zero_sccm": zero_sccm,
    })

co2_hourly = pd.concat(co2_hourly_parts, ignore_index=True)
mask_audit = pd.DataFrame(mask_rows)
display(mask_audit.round(4))

## Referencia M0 en LAB019–LAB021

Se conserva la reproducción del modelo vigente como auditoría. En el ajuste ponderado posterior, LAB019–LAB021 dejan de ser holdout y pasan a formar parte de la calibración de M1/M2 y del reajuste local de M0.

In [ ]:
def activity_bracket(time_h, sugar, ethanol):
    time_h = np.asarray(time_h, dtype=float)
    sugar = np.asarray(sugar, dtype=float)
    ethanol = np.asarray(ethanol, dtype=float)
    valid = np.isfinite(sugar) | np.isfinite(ethanol)
    indices = np.flatnonzero(valid)
    if not len(indices):
        return {"lower_h": 0.0, "upper_h": 0.0, "center_h": 0.0, "scale_h": 1.0}
    s0 = sugar[np.isfinite(sugar)][0] if np.isfinite(sugar).any() else np.nan
    e0 = ethanol[np.isfinite(ethanol)][0] if np.isfinite(ethanol).any() else np.nan
    active = (np.isfinite(sugar) & (sugar <= s0 - 5.0)) | (np.isfinite(ethanol) & (ethanol >= e0 + 2.0))
    active_indices = np.flatnonzero(active)
    if len(active_indices):
        upper_i = int(active_indices[0])
        earlier = indices[indices < upper_i]
        lower_i = int(earlier[-1]) if len(earlier) else upper_i
        lower, upper = float(time_h[lower_i]), float(time_h[upper_i])
    else:
        lower = upper = float(time_h[indices[0]])
    return {"lower_h": lower, "upper_h": upper, "center_h": 0.5 * (lower + upper), "scale_h": max(1.0, (upper - lower) / (2.0 * math.log(9.0))) if upper > lower else 1.0}


co2_predictions = {}
activation_rows = []
for lab in LABS:
    batch = holdout_batches[lab]
    observed = co2_hourly[co2_hourly["lab"].eq(lab)].copy()
    horizon = float(observed["t_h"].max())
    stub = observed[["t_h"]].copy()
    stub["batch"] = lab
    stub["matrix"] = "natural"
    stub["calibratable"] = True
    stub["chemistry_last_h"] = horizon
    caches, _diag = co2_cross.build_driver_cache({lab: batch}, {"natural": theta_full}, stub)
    conditional_cache = caches[lab]

    core = core_predictions[lab]
    predicted_bracket = activity_bracket(core.index, core["G"] + core["F"], core["E"])
    c = chem[chem["lab"].eq(lab)].sort_values("t_h")
    observed_bracket = activity_bracket(c["t_h"], c["GF_g_l"], c["E_g_l"])
    primary_cache = replace(
        conditional_cache,
        chemical_activity_lower_h=predicted_bracket["lower_h"],
        chemical_activity_upper_h=predicted_bracket["upper_h"],
        chemical_activity_center_h=predicted_bracket["center_h"],
        chemical_activity_scale_h=predicted_bracket["scale_h"],
    )
    conditional_cache = replace(
        conditional_cache,
        chemical_activity_lower_h=observed_bracket["lower_h"],
        chemical_activity_upper_h=observed_bracket["upper_h"],
        chemical_activity_center_h=observed_bracket["center_h"],
        chemical_activity_scale_h=observed_bracket["scale_h"],
    )

    for scenario, cache, bracket in (
        ("primary_end_to_end", primary_cache, predicted_bracket),
        ("observed_chemistry_conditioned", conditional_cache, observed_bracket),
    ):
        qgas, qprod, o2, phi = co2_cross.raw_qgas_grid_prediction(
            cache,
            k_release_h=co2_params["kCO2_release_h"], sat_scale=co2_params["CO2sat_scale"],
            o2_qmax_mg_gdw_h=co2_params["O2_qmax_mg_gdw_h"], o2_initial_scale=co2_params["O2_initial_scale"],
            chemistry_aligned=True, nitrogen_boost_transition=True,
            pulse_t_rise_h=co2_params["pulse_t_rise_h"], pulse_activity_gain=co2_params["pulse_activity_gain"],
            continuous_release=True, bounded_chemical_activation=True,
            chem_activation_start_fraction=co2_params["chem_activation_start_fraction"],
            chem_activation_duration_fraction=co2_params["chem_activation_duration_fraction"],
        )
        co2_predictions[(scenario, lab)] = pd.DataFrame({
            "t_h": cache.time_h, "co2_predicted_g_l_h": co2_params["matrix_gain"] * qgas,
            "co2_produced_g_l_h": qprod, "O2_mg_l": o2, "anaerobic_fraction": phi,
        })
        activation_rows.append({"scenario": scenario, "lab": lab, **bracket})

activation_table = pd.DataFrame(activation_rows)
display(activation_table.round(3))

In [ ]:
def sustained_onset(time_h, values, threshold, consecutive=3):
    t, y = np.asarray(time_h, float), np.asarray(values, float)
    above = np.isfinite(y) & (y >= threshold)
    for i in range(0, len(y) - consecutive + 1):
        if above[i:i + consecutive].all():
            return float(t[i])
    return np.nan


prediction_rows, metric_rows = [], []
for lab in LABS:
    obs_all = co2_hourly[(co2_hourly["lab"].eq(lab)) & co2_hourly["co2_valid"]].sort_values("t_h").copy()
    early = obs_all[obs_all["t_h"].le(12.0)]
    baseline = float(early["co2_observed_g_l_h"].median()) if len(early) else 0.0
    peak = float(obs_all["co2_observed_g_l_h"].max())
    onset_threshold = max(co2_cross.CO2_DETECTION_LIMIT_G_L_H, baseline + 0.10 * max(peak - baseline, 0.0))
    for scenario in ("primary_end_to_end", "observed_chemistry_conditioned"):
        pred_curve = co2_predictions[(scenario, lab)]
        obs = obs_all.copy()
        obs["predicted"] = np.interp(obs["t_h"], pred_curve["t_h"], pred_curve["co2_predicted_g_l_h"])
        obs["scenario"] = scenario
        prediction_rows.append(obs[["scenario", "lab", "timestamp", "t_h", "co2_observed_g_l_h", "predicted", "left_censored"]])
        for domain, candidate in (
            ("pre_pulse2", obs["t_h"] < PULSE2_H[lab]),
            ("post_pulse2", obs["t_h"] >= PULSE2_H[lab]),
            ("full", np.ones(len(obs), dtype=bool)),
        ):
            part = obs[candidate].copy()
            if part.empty:
                continue
            error = part["predicted"].to_numpy() - part["co2_observed_g_l_h"].to_numpy()
            observed_values = part["co2_observed_g_l_h"].to_numpy()
            predicted_values = part["predicted"].to_numpy()
            times = part["t_h"].to_numpy()
            denom = np.sum((observed_values - observed_values.mean()) ** 2)
            dt = np.diff(times)
            pair_ok = (dt > 0) & (dt <= 1.5)
            auc_obs = float(np.sum(0.5 * (observed_values[:-1] + observed_values[1:]) * dt * pair_ok))
            auc_pred = float(np.sum(0.5 * (predicted_values[:-1] + predicted_values[1:]) * dt * pair_ok))
            onset_obs = sustained_onset(times, observed_values, onset_threshold)
            onset_pred = sustained_onset(times, predicted_values, onset_threshold)
            metric_rows.append({
                "scenario": scenario, "lab": lab, "domain": domain, "n": len(part),
                "MAE_g_l_h": float(np.mean(np.abs(error))), "RMSE_g_l_h": float(np.sqrt(np.mean(error**2))),
                "bias_pred_minus_obs_g_l_h": float(np.mean(error)),
                "R2": float(1.0 - np.sum(error**2) / denom) if denom > 0 else np.nan,
                "onset_threshold_g_l_h": onset_threshold, "onset_observed_h": onset_obs, "onset_predicted_h": onset_pred,
                "onset_error_h": onset_pred - onset_obs if np.isfinite(onset_obs) and np.isfinite(onset_pred) else np.nan,
                "peak_observed_g_l_h": float(observed_values.max()), "peak_predicted_g_l_h": float(predicted_values.max()),
                "peak_time_observed_h": float(times[np.argmax(observed_values)]), "peak_time_predicted_h": float(times[np.argmax(predicted_values)]),
                "AUC_observed_g_l": auc_obs, "AUC_predicted_g_l": auc_pred, "AUC_error_g_l": auc_pred - auc_obs,
            })

co2_prediction_rows = pd.concat(prediction_rows, ignore_index=True)
co2_metrics = pd.DataFrame(metric_rows)
display(co2_metrics[co2_metrics["domain"].eq("full")].round(4))

## Entrada causal sin bracket observado

La entrada `qprod` conserva la cinética, el bloque de O2 y la transición del pulso, pero desactiva el gate construido con química futura.

In [ ]:
from scipy import signal
from scipy.optimize import least_squares

HISTORICAL_CO2_DIR = (
    LAB2026 / "results" /
    "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010"
)
historical_obs = pd.read_csv(HISTORICAL_CO2_DIR / "co2_observations_hourly.csv")
historical_obs = historical_obs[historical_obs["matrix"].eq("natural")].copy()
for column in ("calibratable", "left_censored"):
    historical_obs[column] = historical_obs[column].map(
        lambda value: value if isinstance(value, bool) else str(value).strip().lower() == "true"
    )

historical_chemistry, historical_natural_batches, _, historical_natural_tables = (
    natural_loader.make_medium_batches("natural")
)
historical_schedule = co2_cross.natural_nutrient_pulse_schedule(
    historical_chemistry, historical_natural_tables["metadata"]
)
historical_natural_batches = co2_cross.override_natural_nutrient_pulses(
    historical_natural_batches, historical_schedule
)
historical_batches_all = {batch.batch: batch for batch in historical_natural_batches}
historical_cache, historical_driver_audit = co2_cross.build_driver_cache(
    historical_batches_all,
    {"natural": theta_full},
    historical_obs,
)

HISTORICAL_TRAIN_LABS = ("LAB005", "LAB007", "LAB008")
RECENT_TRAIN_LABS = LABS
TRAIN_LABS = HISTORICAL_TRAIN_LABS + RECENT_TRAIN_LABS
VALIDATION_LABS = ("LAB011", "LAB012")
SENSITIVITY_LABS = ("LAB010",)
FIT_LAB_WEIGHT = {
    **{lab: 1.0 for lab in HISTORICAL_TRAIN_LABS},
    **{lab: 2.0 for lab in RECENT_TRAIN_LABS},
}
assert "LAB004" not in TRAIN_LABS and "LAB006" not in TRAIN_LABS
assert set(HISTORICAL_TRAIN_LABS + VALIDATION_LABS + SENSITIVITY_LABS).issubset(historical_cache)


def ungated_effective_input(cache):
    qprod, o2, phi = co2_cross.effective_qprod_grid(
        cache,
        o2_qmax_mg_gdw_h=co2_params["O2_qmax_mg_gdw_h"],
        o2_initial_scale=co2_params["O2_initial_scale"],
        chemistry_aligned=False,
        nitrogen_boost_transition=True,
        pulse_t_rise_h=co2_params["pulse_t_rise_h"],
        pulse_activity_gain=co2_params["pulse_activity_gain"],
    )
    return np.asarray(qprod, dtype=float)


historical_qprod = {
    lab: ungated_effective_input(cache)
    for lab, cache in historical_cache.items()
}

# Cache para LAB019–LAB021. La química observada no se usa como gate.
recent_cache = {}
recent_qprod = {}
for lab in RECENT_TRAIN_LABS:
    observed = co2_hourly[co2_hourly["lab"].eq(lab)].sort_values("t_h")
    horizon = float(observed["t_h"].max())
    stub = observed[["t_h"]].copy()
    stub["batch"] = lab
    stub["matrix"] = "natural"
    stub["calibratable"] = True
    stub["chemistry_last_h"] = horizon
    caches, _ = co2_cross.build_driver_cache(
        {lab: holdout_batches[lab]}, {"natural": theta_full}, stub
    )
    recent_cache[lab] = caches[lab]
    recent_qprod[lab] = ungated_effective_input(caches[lab])

# Holdout LAB016–LAB018: soporte observado ya auditado y limpio, sin usarlo en el fit.
EXTERNAL_LABS = ("LAB016", "LAB017", "LAB018")
external_rows_all = pd.read_csv(HISTORICAL_CO2_DIR / "external_holdout_prediction_rows_A_vs_B.csv")
external_observed = (
    external_rows_all[["batch", "time_h", "observed_g_l_h"]]
    .drop_duplicates()
    .rename(columns={"batch": "lab", "observed_g_l_h": "co2_observed_g_l_h"})
    .sort_values(["lab", "time_h"])
    .reset_index(drop=True)
)
assert set(external_observed["lab"]) == set(EXTERNAL_LABS)
external_support_audit = pd.read_csv(HISTORICAL_CO2_DIR / "external_holdout_support_LAB016_018.csv")

external_initial_reference = pd.read_csv(
    HISTORICAL_CO2_DIR / "external_kinetic_holdout_initial_conditions_LAB013_015.csv"
)
inherited_x0 = float(external_initial_reference["X"].mean())
inherited_xd0 = float(external_initial_reference["Xd"].mean())
external_initial_sheet = pd.read_excel(
    DATA_016_018_DIR / "LAB016-018-offline-measurements.xlsx",
    sheet_name="CondicionesIniciales",
)
external_initial_sheet = external_initial_sheet.set_index("CodigoLab")

external_temp_files = {
    "LAB016": "Temp_F1_LAB016.csv",
    "LAB017": "Temp_F2_LAB017.csv",
    "LAB018": "Temp_F3._LAB018.csv",
}
external_co2_files = {
    "LAB016": "CO2_FILT_F1_LAB016.csv",
    "LAB017": "CO2_FILT_F2_LAB017.csv",
    "LAB018": "CO2_FILT_F3_LAB018.csv",
}
external_batches = {}
external_cache = {}
external_qprod = {}
external_m0 = {}
external_initial_rows = []
external_full_observed_parts = []

for lab in EXTERNAL_LABS:
    init = external_initial_sheet.loc[lab]
    t0 = pd.Timestamp(init["Fecha"]).normalize() + pd.to_timedelta(str(init["Hora"]))
    support = external_observed[external_observed["lab"].eq(lab)].sort_values("time_h")
    strict_holdout_last_h = float(support["time_h"].max())

    full_co2 = pd.read_csv(DATA_016_018_DIR / external_co2_files[lab], parse_dates=["timestamp"])
    full_co2["t_h"] = (full_co2["timestamp"] - t0).dt.total_seconds() / 3600.0
    baseline_support = full_co2[full_co2["t_h"].between(-2.0, -0.5)]["flow_filt_sccm"].dropna()
    baseline_sccm = float(baseline_support.median()) if len(baseline_support) else 0.0
    full_co2["flow_corrected_sccm"] = np.maximum(full_co2["flow_filt_sccm"] - baseline_sccm, 0.0)
    full_hourly = (
        full_co2.set_index("timestamp")[["flow_corrected_sccm"]]
        .resample("1h").median().dropna().reset_index()
    )
    full_hourly["t_h"] = (full_hourly["timestamp"] - t0).dt.total_seconds() / 3600.0
    full_hourly = full_hourly[full_hourly["t_h"].ge(0.0)].copy()
    full_hourly["lab"] = lab
    full_hourly["co2_observed_g_l_h"] = full_hourly["flow_corrected_sccm"] * CO2_FACTOR_G_L_H_PER_SCCM
    external_full_observed_parts.append(full_hourly)

    horizon = float(full_hourly["t_h"].max())
    grid_end = math.ceil(horizon / GRID_H) * GRID_H
    grid = np.arange(0.0, grid_end + GRID_H / 2.0, GRID_H)

    temp = pd.read_csv(DATA_016_018_DIR / external_temp_files[lab], parse_dates=["timestamp"])
    temp["time_h"] = (temp["timestamp"] - t0).dt.total_seconds() / 3600.0
    temp = temp.dropna(subset=["T"]).sort_values("time_h")
    temperature_grid = np.interp(
        grid, temp["time_h"].to_numpy(dtype=float), temp["T"].to_numpy(dtype=float)
    )
    initials = {
        "X": inherited_x0, "Xd": inherited_xd0,
        "N": float(init["YAN_mg_L"]) * 1e-3,
        "G": float(init["Glucosa_g_L"]), "F": float(init["Fructosa_g_L"]),
        "E": 0.0, "Gly": float(init["Glicerol_g_L"]),
    }
    observations = {state: np.full(len(grid), np.nan) for state in base.STATE_NAMES}
    batch = base.BatchData(
        medium="natural", batch=lab, time=grid, temperature_c=temperature_grid,
        pulses={channel: tuple() for channel in base.INPUT_CHANNELS},
        observations=observations, initials=initials,
    )
    core = base.simulate(batch, theta_full, grid)
    assert core is not None and np.isfinite(core.to_numpy(dtype=float)).all()
    stub = full_hourly[["t_h"]].copy()
    stub["batch"] = lab
    stub["matrix"] = "natural"
    stub["calibratable"] = True
    stub["chemistry_last_h"] = horizon
    caches, _ = co2_cross.build_driver_cache({lab: batch}, {"natural": theta_full}, stub)
    cache = caches[lab]
    external_batches[lab] = batch
    external_cache[lab] = cache
    external_qprod[lab] = ungated_effective_input(cache)

    qgas, _, _, _ = co2_cross.raw_qgas_grid_prediction(
        cache,
        k_release_h=co2_params["kCO2_release_h"],
        sat_scale=co2_params["CO2sat_scale"],
        o2_qmax_mg_gdw_h=co2_params["O2_qmax_mg_gdw_h"],
        o2_initial_scale=co2_params["O2_initial_scale"],
        chemistry_aligned=True,
        nitrogen_boost_transition=True,
        pulse_t_rise_h=co2_params["pulse_t_rise_h"],
        pulse_activity_gain=co2_params["pulse_activity_gain"],
        continuous_release=True,
        bounded_chemical_activation=True,
        chem_activation_start_fraction=co2_params["chem_activation_start_fraction"],
        chem_activation_duration_fraction=co2_params["chem_activation_duration_fraction"],
    )
    external_m0[lab] = co2_params["matrix_gain"] * qgas
    external_initial_rows.append({
        "lab": lab, **initials,
        "X_Xd_source": "media LAB013–LAB015 (mismo mosto)",
        "G_F_N_Gly_source": str(init["Fuente"]),
        "strict_holdout_last_h": strict_holdout_last_h, "record_last_h": horizon,
    })

external_full_observed = pd.concat(external_full_observed_parts, ignore_index=True)
external_initial_table = pd.DataFrame(external_initial_rows)
inventory = pd.DataFrame([
    {"grupo": "calibración histórica", "labs": ", ".join(HISTORICAL_TRAIN_LABS), "peso_objetivo_por_lab": 1.0, "participa_fit": True},
    {"grupo": "calibración reciente prioritaria", "labs": ", ".join(RECENT_TRAIN_LABS), "peso_objetivo_por_lab": 2.0, "participa_fit": True},
    {"grupo": "validación para selección M1/M2", "labs": ", ".join(VALIDATION_LABS), "peso_objetivo_por_lab": np.nan, "participa_fit": False},
    {"grupo": "holdout externo", "labs": ", ".join(EXTERNAL_LABS), "peso_objetivo_por_lab": np.nan, "participa_fit": False},
])
display(inventory)
display(external_initial_table)
display(external_support_audit)

## Ajuste ponderado de M1 y M2

M1 usa un polo, `dR/dt = qprod - R/tau`; M2 usa dos polos causales en cascada. `theta_full` permanece congelado. Cada LAB aporta un bloque normalizado por cantidad de puntos y amplitud. LAB019–LAB021 reciben peso 2 en la suma de cuadrados; LAB005/LAB007/LAB008 reciben peso 1. Para los tres LAB recientes se usa sólo el tramo válido anterior al segundo pulso, de modo que el ajuste del onset no absorba incertidumbre de la respuesta nutricional tardía.

In [ ]:
TAU_BOUNDS_H = (0.25, 96.0)


def simulate_effective_filter(time_h, qprod, model, taus_h):
    time_h = np.asarray(time_h, dtype=float)
    qprod = np.asarray(qprod, dtype=float)
    dt = np.diff(time_h)
    assert len(time_h) == len(qprod) and np.all(dt > 0)
    taus = np.sort(np.asarray(taus_h, dtype=float))
    if model == "M1":
        denominator = [taus[0], 1.0]
    elif model == "M2":
        denominator = [taus[0] * taus[1], taus[0] + taus[1], 1.0]
    else:
        raise ValueError(model)

    # Integracion causal ZOH intervalo a intervalo; admite grillas regulares o mixtas.
    A, B, C, D = signal.tf2ss([1.0], denominator)
    state = np.zeros(A.shape[0], dtype=float)
    output = np.empty(len(time_h), dtype=float)
    output[0] = float((C @ state.reshape(-1, 1) + D * qprod[0]).squeeze())
    discrete_cache = {}
    for index, step_h in enumerate(dt):
        key = round(float(step_h), 12)
        if key not in discrete_cache:
            Ad, Bd, _, _, _ = signal.cont2discrete((A, B, C, D), float(step_h), method="zoh")
            discrete_cache[key] = (Ad, Bd[:, 0])
        Ad, Bd = discrete_cache[key]
        state = Ad @ state + Bd * qprod[index]
        output[index + 1] = float((C @ state.reshape(-1, 1) + D * qprod[index + 1]).squeeze())
    return np.maximum(output, 0.0)


def historical_support(lab):
    cache = historical_cache[lab]
    group = historical_obs[historical_obs["batch"].eq(lab)].sort_values("t_h").copy()
    if np.isfinite(cache.n_pulse_time_h):
        group = group[group["t_h"] < cache.n_pulse_time_h].copy()
    return group


def recent_support(lab):
    group = co2_hourly[
        co2_hourly["lab"].eq(lab) & co2_hourly["co2_valid"]
        & (co2_hourly["t_h"] < PULSE2_H[lab])
    ].sort_values("t_h").copy()
    group = group.rename(columns={"co2_observed_g_l_h": "co2_rate_g_l_h"})
    group["detection_limit_g_l_h"] = co2_cross.CO2_DETECTION_LIMIT_G_L_H
    return group


def fit_cache_and_input(lab):
    if lab in recent_cache:
        return recent_cache[lab], recent_qprod[lab]
    return historical_cache[lab], historical_qprod[lab]


def fit_support(lab):
    return recent_support(lab) if lab in RECENT_TRAIN_LABS else historical_support(lab)


def onset_threshold(group, observed_column="co2_rate_g_l_h"):
    early = group[group["t_h"] <= 12.0]
    baseline = float(early[observed_column].median()) if len(early) else 0.0
    peak = float(group[observed_column].max())
    detection = (
        float(group["detection_limit_g_l_h"].median())
        if "detection_limit_g_l_h" in group else co2_cross.CO2_DETECTION_LIMIT_G_L_H
    )
    return max(detection, baseline + 0.10 * max(peak - baseline, 0.0))


def shape_residual(log_taus, model, labs):
    taus = np.sort(np.exp(log_taus))
    residuals = []
    for lab in labs:
        cache, qprod = fit_cache_and_input(lab)
        group = fit_support(lab)
        raw_grid = simulate_effective_filter(cache.time_h, qprod, model, taus)
        predicted = np.interp(group["t_h"], cache.time_h, raw_grid)
        observed = group["co2_rate_g_l_h"].to_numpy(dtype=float)
        observed_scale = max(float(np.max(observed)), 1e-12)
        predicted_scale = max(float(np.max(predicted)), 1e-12)
        observed_n = observed / observed_scale
        predicted_n = predicted / predicted_scale
        early_objective_weight = np.where(group["t_h"].to_numpy(dtype=float) <= 48.0, 2.0, 1.0)
        block = np.sqrt(early_objective_weight) * (predicted_n - observed_n)
        threshold_n = onset_threshold(group) / observed_scale
        onset_obs = sustained_onset(group["t_h"], observed_n, threshold_n)
        onset_pred = sustained_onset(group["t_h"], predicted_n, threshold_n)
        if not np.isfinite(onset_pred):
            onset_pred = float(group["t_h"].max()) + 12.0
        onset_residual = (onset_pred - onset_obs) / 12.0 if np.isfinite(onset_obs) else 0.0
        lab_scale = math.sqrt(FIT_LAB_WEIGHT[lab] / (len(block) + 1.0))
        residuals.append(lab_scale * np.r_[block, onset_residual])
    return np.concatenate(residuals)


def fit_timing(model):
    starts = {
        "M1": [(4.0,), (12.0,), (36.0,), (72.0,)],
        "M2": [(3.0, 12.0), (8.0, 24.0), (15.0, 15.0), (6.0, 48.0), (24.0, 48.0)],
    }[model]
    best = None
    rows = []
    for index, start in enumerate(starts):
        result = least_squares(
            lambda values: shape_residual(values, model, TRAIN_LABS),
            np.log(start),
            bounds=(
                np.log([TAU_BOUNDS_H[0]] * len(start)),
                np.log([TAU_BOUNDS_H[1]] * len(start)),
            ),
            max_nfev=400,
        )
        taus = np.sort(np.exp(result.x))
        objective = float(np.dot(result.fun, result.fun))
        rows.append({"model": model, "start": index, "objective_shape": objective, **{f"tau_{i+1}_h": value for i, value in enumerate(taus)}})
        if best is None or objective < best[0]:
            best = (objective, taus)
    return best[1], pd.DataFrame(rows)


def profile_common_gain(model, taus, labs=TRAIN_LABS):
    numerator = 0.0
    denominator = 0.0
    for lab in labs:
        cache, qprod = fit_cache_and_input(lab)
        group = fit_support(lab)
        raw_grid = simulate_effective_filter(cache.time_h, qprod, model, taus)
        predicted = np.interp(group["t_h"], cache.time_h, raw_grid)
        observed = group["co2_rate_g_l_h"].to_numpy(dtype=float)
        keep = ~group["left_censored"].astype(bool).to_numpy()
        predicted = predicted[keep]
        observed = observed[keep]
        relative_scale = max(float(np.max(observed)), co2_cross.CO2_DETECTION_LIMIT_G_L_H)
        point_weight = math.sqrt(FIT_LAB_WEIGHT[lab] / max(len(observed), 1)) / relative_scale
        numerator += float(np.sum(point_weight**2 * predicted * observed))
        denominator += float(np.sum(point_weight**2 * predicted**2))
    return float(np.clip(numerator / max(denominator, 1e-18), 0.05, 20.0))


effective_fits = {}
fit_start_tables = []
for model in ("M1", "M2"):
    taus, starts = fit_timing(model)
    gain = profile_common_gain(model, taus)
    effective_fits[model] = {"taus_h": taus, "gain": gain}
    fit_start_tables.append(starts)

fit_start_table = pd.concat(fit_start_tables, ignore_index=True)
parameter_rows = []
for model, values in effective_fits.items():
    for index, tau in enumerate(values["taus_h"], start=1):
        parameter_rows.append({
            "model": model, "parameter": f"tau_{index}_h", "estimate": tau,
            "lower_bound": TAU_BOUNDS_H[0], "upper_bound": TAU_BOUNDS_H[1],
            "active_bound": min(abs(tau - TAU_BOUNDS_H[0]), abs(tau - TAU_BOUNDS_H[1])) <= 0.01 * (TAU_BOUNDS_H[1] - TAU_BOUNDS_H[0]),
        })
    parameter_rows.append({"model": model, "parameter": "gain", "estimate": values["gain"], "lower_bound": 0.05, "upper_bound": 20.0, "active_bound": False})
effective_parameters = pd.DataFrame(parameter_rows)
display(effective_parameters.round(5))
display(fit_start_table.round(5))

In [ ]:
def trapezoid_on_contiguous_support(time_h, values):
    time_h = np.asarray(time_h, dtype=float)
    values = np.asarray(values, dtype=float)
    dt = np.diff(time_h)
    keep = (dt > 0.0) & (dt <= 1.5)
    return float(np.sum(0.5 * (values[:-1] + values[1:]) * dt * keep))


def metric_row(model, lab, group, predicted):
    observed = (
        group["co2_observed_g_l_h"].to_numpy(dtype=float)
        if "co2_observed_g_l_h" in group else group["co2_rate_g_l_h"].to_numpy(dtype=float)
    )
    time_h = group["t_h"].to_numpy(dtype=float) if "t_h" in group else group["time_h"].to_numpy(dtype=float)
    threshold_group = group.rename(columns={"co2_observed_g_l_h": "co2_rate_g_l_h", "time_h": "t_h"})
    threshold = onset_threshold(threshold_group)
    onset_obs = sustained_onset(time_h, observed, threshold)
    onset_pred = sustained_onset(time_h, predicted, threshold)
    error = predicted - observed
    peak_obs = float(np.max(observed))
    peak_pred = float(np.max(predicted))
    auc_obs = trapezoid_on_contiguous_support(time_h, observed)
    auc_pred = trapezoid_on_contiguous_support(time_h, predicted)
    return {
        "model": model, "lab": lab, "n": len(group),
        "RMSE_g_l_h": float(np.sqrt(np.mean(error**2))),
        "NRMSE_peak": float(np.sqrt(np.mean(error**2)) / max(peak_obs, 1e-12)),
        "onset_threshold_g_l_h": threshold,
        "onset_observed_h": onset_obs, "onset_predicted_h": onset_pred,
        "onset_error_h": onset_pred - onset_obs if np.isfinite(onset_obs) and np.isfinite(onset_pred) else np.nan,
        "peak_observed_g_l_h": peak_obs, "peak_predicted_g_l_h": peak_pred,
        "peak_time_observed_h": float(time_h[np.argmax(observed)]),
        "peak_time_predicted_h": float(time_h[np.argmax(predicted)]),
        "AUC_observed_g_l": auc_obs, "AUC_predicted_g_l": auc_pred,
        "AUC_relative_error": (auc_pred - auc_obs) / max(auc_obs, 1e-12),
    }


fit_prediction_rows = []
fit_metric_rows = []
evaluation_groups = (
    ("train_historical", HISTORICAL_TRAIN_LABS),
    ("train_recent_weight_2", RECENT_TRAIN_LABS),
    ("historical_validation", VALIDATION_LABS),
    ("LAB010_sensitivity", SENSITIVITY_LABS),
)
for role, labs in evaluation_groups:
    for lab in labs:
        if lab in RECENT_TRAIN_LABS:
            cache, qprod, group = recent_cache[lab], recent_qprod[lab], recent_support(lab)
        else:
            cache, qprod, group = historical_cache[lab], historical_qprod[lab], historical_support(lab)
        for model, values in effective_fits.items():
            raw = simulate_effective_filter(cache.time_h, qprod, model, values["taus_h"])
            predicted = values["gain"] * np.interp(group["t_h"], cache.time_h, raw)
            row = metric_row(model, lab, group, predicted)
            row["role"] = role
            fit_metric_rows.append(row)
            fit_prediction_rows.append(pd.DataFrame({
                "role": role, "model": model, "lab": lab,
                "t_h": group["t_h"].to_numpy(dtype=float),
                "observed": group["co2_rate_g_l_h"].to_numpy(dtype=float),
                "predicted": predicted,
            }))

fit_metrics_effective = pd.DataFrame(fit_metric_rows)
fit_predictions_effective = pd.concat(fit_prediction_rows, ignore_index=True)
validation_selection = (
    fit_metrics_effective[fit_metrics_effective["role"].eq("historical_validation")]
    .groupby("model", as_index=False)
    .agg(
        median_abs_onset_error_h=("onset_error_h", lambda values: float(np.nanmedian(np.abs(values)))),
        median_NRMSE_peak=("NRMSE_peak", "median"),
    )
)
validation_selection["selection_score"] = (
    validation_selection["median_abs_onset_error_h"]
    + 24.0 * validation_selection["median_NRMSE_peak"]
)
SELECTED_MODEL = str(validation_selection.sort_values("selection_score").iloc[0]["model"])
display(fit_metrics_effective.round(4))
display(validation_selection.round(4))
print("Modelo seleccionado con LAB011–LAB012 antes del holdout externo:", SELECTED_MODEL)

In [ ]:
def complete_observed_curve(lab):
    if lab in RECENT_TRAIN_LABS:
        group = co2_hourly[
            co2_hourly["lab"].eq(lab) & co2_hourly["co2_valid"]
        ].sort_values("t_h")
        return group["t_h"].to_numpy(dtype=float), group["co2_observed_g_l_h"].to_numpy(dtype=float)
    group = historical_obs[historical_obs["batch"].eq(lab)].sort_values("t_h")
    return group["t_h"].to_numpy(dtype=float), group["co2_rate_g_l_h"].to_numpy(dtype=float)


fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.2), layout="constrained")
for index, (ax, lab) in enumerate(zip(axes.ravel(), TRAIN_LABS)):
    time_obs, observed = complete_observed_curve(lab)
    cache, qprod = fit_cache_and_input(lab)
    ax.plot(time_obs, observed, color="black", lw=1.7, label="Observado completo")
    for model, color in (("M1", "#0072B2"), ("M2", "#D55E00")):
        values = effective_fits[model]
        predicted = values["gain"] * simulate_effective_filter(cache.time_h, qprod, model, values["taus_h"])
        ax.plot(cache.time_h, predicted, color=color, lw=2.0, label=model)
    ax.set_title(f"{lab} · peso {FIT_LAB_WEIGHT[lab]:.0f}", fontsize=11)
    if index // 3 == 1:
        ax.set_xlabel("t desde inoculación [h]")
    if index % 3 == 0:
        ax.set_ylabel(r"Flujo de CO$_2$ [g L$^{-1}$ h$^{-1}$]")
    ax.grid(alpha=0.25)
axes[0, 0].legend(loc="upper right", frameon=True, fontsize=8.0)
fig.suptitle("Ajuste ponderado de la capa de CO2 · curvas completas", fontsize=14)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), layout="constrained")
for index, (ax, lab) in enumerate(zip(axes, VALIDATION_LABS)):
    time_obs, observed = complete_observed_curve(lab)
    cache, qprod = historical_cache[lab], historical_qprod[lab]
    ax.plot(time_obs, observed, color="black", lw=1.7, label="Observado completo")
    for model, color in (("M1", "#0072B2"), ("M2", "#D55E00")):
        values = effective_fits[model]
        predicted = values["gain"] * simulate_effective_filter(cache.time_h, qprod, model, values["taus_h"])
        width = 2.5 if model == SELECTED_MODEL else 1.6
        ax.plot(cache.time_h, predicted, color=color, lw=width, label=model)
    ax.set_title(lab)
    ax.set_xlabel("t desde inoculación [h]")
    if index == 0:
        ax.set_ylabel(r"Flujo de CO$_2$ [g L$^{-1}$ h$^{-1}$]")
        ax.legend(loc="upper right", frameon=True, fontsize=8.0)
    ax.grid(alpha=0.25)
fig.suptitle("Selección de estructura en LAB011–LAB012 · curvas completas", fontsize=13)
display(fig)
plt.close(fig)

## Reajuste local de M0 con los mismos lotes y pesos

El modelo vigente M0 también se recalibra con la misma data y los mismos pesos del ajuste de M1/M2: LAB005/LAB007/LAB008 con peso 1 y LAB019–LAB021 con peso 2, sobre los mismos tramos de ajuste (pre-pulso histórico; pre-segundo-pulso válido en los recientes), sin LAB004 ni LAB006. Se reutiliza sin cambios la maquinaria original de calibración (`co2_cross.fit_matrix`): los 8 parámetros de forma se optimizan en escala logarítmica con `matrix_gain` perfilada, censura izquierda y residuos de onset y de pico post-pulso. El peso 2 se implementa duplicando cada lote reciente como lote gemelo en la suma de cuadrados, por lo que el ajuste reporta 9 lotes (6 únicos). En LAB016–LAB018 la compuerta química es degenerada (sin química observada, la activación es 1 desde t=0), de modo que allí M0 actual y reajustado difieren sólo en los parámetros restantes. `theta_full` y `three_state_co2_parameters.csv` siguen congelados: este reajuste vive únicamente en este notebook.


In [ ]:
M0_REFIT_ALIAS_SUFFIX = "__w2"


def m0_refit_frame(lab):
    # Frame de ajuste con exactamente el tramo usado por M1/M2 para ese lote.
    if lab in RECENT_TRAIN_LABS:
        group = recent_support(lab)
        sigma_floor = float(co2_cross.CO2_RESIDUAL_SIGMA_FLOOR_G_L_H)
        base_cache = recent_cache[lab]
    else:
        group = historical_support(lab)
        sigma_floor = float(group["residual_sigma_floor_g_l_h"].iloc[0])
        base_cache = historical_cache[lab]
    frame = pd.DataFrame({
        "batch": lab,
        "matrix": "natural",
        "calibratable": True,
        "t_h": group["t_h"].to_numpy(dtype=float),
        "co2_rate_g_l_h": group["co2_rate_g_l_h"].to_numpy(dtype=float),
        "left_censored": group["left_censored"].astype(bool).to_numpy(),
        "detection_limit_g_l_h": group["detection_limit_g_l_h"].to_numpy(dtype=float),
        "residual_sigma_floor_g_l_h": np.full(len(group), sigma_floor),
    }).sort_values("t_h").reset_index(drop=True)
    # El cache de ajuste comparte la trayectoria del cache completo; sólo se
    # reemplazan los tiempos de observación, alineados 1:1 con el frame.
    fit_cache = replace(base_cache, observation_time_h=frame["t_h"].to_numpy(dtype=float))
    return frame, fit_cache


m0_refit_frames, m0_refit_caches = {}, {}
for lab in TRAIN_LABS:
    frame, fit_cache = m0_refit_frame(lab)
    m0_refit_frames[lab] = frame
    m0_refit_caches[lab] = fit_cache
    if FIT_LAB_WEIGHT[lab] > 1.0:
        alias = f"{lab}{M0_REFIT_ALIAS_SUFFIX}"
        m0_refit_frames[alias] = frame.assign(batch=alias)
        m0_refit_caches[alias] = replace(fit_cache, batch=alias)

m0_refit_observations = pd.concat(m0_refit_frames.values(), ignore_index=True)
m0_refit_inventory = pd.DataFrame([
    {
        "lote": lab, "peso": FIT_LAB_WEIGHT[lab], "n_puntos": len(m0_refit_frames[lab]),
        "t_h_final": float(m0_refit_frames[lab]["t_h"].max()),
    }
    for lab in TRAIN_LABS
])
display(m0_refit_inventory)

# Reutiliza la maquinaria original de calibración; el peso 2 queda implementado
# como lote gemelo dentro de la misma suma de cuadrados. El primer start es el
# valor de producción actual de three_state_co2_parameters.csv.
m0_refit_fit, m0_refit_start_table = co2_cross.fit_matrix(
    "natural",
    m0_refit_observations,
    m0_refit_caches,
    n_starts=5,
    max_nfev=300,
    initial_parameters=co2_params,
)
m0_refit_parameters = {name: float(m0_refit_fit[name]) for name in co2_cross.PARAMETER_BOUNDS}

m0_refit_compare = pd.DataFrame([
    {
        "parameter": name,
        "M0_actual": float(co2_params[name]),
        "M0_refit": m0_refit_parameters[name],
        "change_pct": 100.0 * (m0_refit_parameters[name] / float(co2_params[name]) - 1.0),
        "lower_bound": lower,
        "upper_bound": upper,
        "active_bound": min(
            abs(m0_refit_parameters[name] - lower), abs(m0_refit_parameters[name] - upper)
        ) <= 0.01 * (upper - lower),
    }
    for name, (lower, upper) in co2_cross.PARAMETER_BOUNDS.items()
])
display(m0_refit_compare.round(4))
display(m0_refit_start_table[["start", "success", "nfev", "wsse_equal_batch", "matrix_gain"]].round(4))


In [ ]:
def m0_grid_prediction(cache, params):
    # Predicción M0 completa: compuerta química, rampa causal de N y liberación continua.
    qgas, _, _, _ = co2_cross.raw_qgas_grid_prediction(
        cache,
        k_release_h=params["kCO2_release_h"],
        sat_scale=params["CO2sat_scale"],
        o2_qmax_mg_gdw_h=params["O2_qmax_mg_gdw_h"],
        o2_initial_scale=params["O2_initial_scale"],
        chemistry_aligned=True,
        nitrogen_boost_transition=True,
        pulse_t_rise_h=params["pulse_t_rise_h"],
        pulse_activity_gain=params["pulse_activity_gain"],
        continuous_release=True,
        bounded_chemical_activation=True,
        chem_activation_start_fraction=params["chem_activation_start_fraction"],
        chem_activation_duration_fraction=params["chem_activation_duration_fraction"],
    )
    return params["matrix_gain"] * qgas


m0_refit_metric_rows, m0_refit_prediction_rows = [], []
for role, labs in evaluation_groups:
    for lab in labs:
        if lab in RECENT_TRAIN_LABS:
            cache, group = recent_cache[lab], recent_support(lab)
        else:
            cache, group = historical_cache[lab], historical_support(lab)
        for model_name, params in (("M0_actual", co2_params), ("M0_refit", m0_refit_parameters)):
            predicted = np.interp(group["t_h"], cache.time_h, m0_grid_prediction(cache, params))
            row = metric_row(model_name, lab, group, predicted)
            row["role"] = role
            m0_refit_metric_rows.append(row)
            m0_refit_prediction_rows.append(pd.DataFrame({
                "role": role, "model": model_name, "lab": lab,
                "t_h": group["t_h"].to_numpy(dtype=float),
                "observed": group["co2_rate_g_l_h"].to_numpy(dtype=float),
                "predicted": predicted,
            }))

m0_refit_metrics = pd.DataFrame(m0_refit_metric_rows)
m0_refit_predictions = pd.concat(m0_refit_prediction_rows, ignore_index=True)

# Selección comparable sobre LAB011-LAB012 con la misma fórmula de la tabla de M1/M2.
validation_m0_selection = (
    pd.concat([
        m0_refit_metrics[m0_refit_metrics["role"].eq("historical_validation")],
        fit_metrics_effective[fit_metrics_effective["role"].eq("historical_validation")],
    ], ignore_index=True)
    .groupby("model", as_index=False)
    .agg(
        median_abs_onset_error_h=("onset_error_h", lambda values: float(np.nanmedian(np.abs(values)))),
        median_NRMSE_peak=("NRMSE_peak", "median"),
    )
)
validation_m0_selection["selection_score"] = (
    validation_m0_selection["median_abs_onset_error_h"]
    + 24.0 * validation_m0_selection["median_NRMSE_peak"]
)
display(m0_refit_metrics[[
    "model", "lab", "role", "n", "RMSE_g_l_h", "NRMSE_peak",
    "onset_error_h", "peak_time_predicted_h", "AUC_relative_error",
]].round(4))
display(validation_m0_selection.round(4))

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.2), layout="constrained")
for index, (ax, lab) in enumerate(zip(axes.ravel(), TRAIN_LABS)):
    time_obs, observed = complete_observed_curve(lab)
    cache = recent_cache[lab] if lab in RECENT_TRAIN_LABS else historical_cache[lab]
    ax.plot(time_obs, observed, color="black", lw=1.7, label="Observado completo")
    ax.plot(cache.time_h, m0_grid_prediction(cache, co2_params), color="0.55", ls="--", lw=1.7, label="M0 actual")
    ax.plot(cache.time_h, m0_grid_prediction(cache, m0_refit_parameters), color="#009E73", lw=2.0, label="M0 reajustado")
    ax.set_title(f"{lab} · peso {FIT_LAB_WEIGHT[lab]:.0f}", fontsize=11)
    if index // 3 == 1:
        ax.set_xlabel("t desde inoculación [h]")
    if index % 3 == 0:
        ax.set_ylabel(r"Flujo de CO$_2$ [g L$^{-1}$ h$^{-1}$]")
    ax.grid(alpha=0.25)
axes[0, 0].legend(loc="upper right", frameon=True, fontsize=8.0)
fig.suptitle("Reajuste local de M0 con los mismos lotes y pesos · curvas completas", fontsize=14)
display(fig)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), layout="constrained")
for index, (ax, lab) in enumerate(zip(axes, VALIDATION_LABS)):
    time_obs, observed = complete_observed_curve(lab)
    cache = historical_cache[lab]
    ax.plot(time_obs, observed, color="black", lw=1.7, label="Observado completo")
    ax.plot(cache.time_h, m0_grid_prediction(cache, co2_params), color="0.55", ls="--", lw=1.7, label="M0 actual")
    ax.plot(cache.time_h, m0_grid_prediction(cache, m0_refit_parameters), color="#009E73", lw=2.0, label="M0 reajustado")
    ax.set_title(lab)
    ax.set_xlabel("t desde inoculación [h]")
    if index == 0:
        ax.set_ylabel(r"Flujo de CO$_2$ [g L$^{-1}$ h$^{-1}$]")
        ax.legend(loc="upper right", frameon=True, fontsize=8.0)
    ax.grid(alpha=0.25)
fig.suptitle("M0 actual vs reajustado en LAB011–LAB012 · curvas completas", fontsize=13)
display(fig)
plt.close(fig)


## Holdout externo LAB016–LAB018

Las condiciones G0, F0, N0 y Gly0 provienen de la hoja CondicionesIniciales, que las documenta como el promedio heredado de LAB013–LAB015 del mismo mosto. X0 y Xd0 son también las medias de LAB013–LAB015. Las métricas utilizan únicamente el soporte limpio anterior al primer pulso nutricional posible, mientras que la figura muestra el registro completo. LAB016–LAB018 no intervienen en la estimación de tau, ganancia, selección entre M1/M2 ni en el reajuste de M0.

In [ ]:
candidate_curves = {}
holdout_metric_rows = []

for lab in EXTERNAL_LABS:
    group = external_observed[external_observed["lab"].eq(lab)].sort_values("time_h").copy()
    group = group.rename(columns={"time_h": "t_h"})
    cache = external_cache[lab]

    candidate_curves[("M0_actual", lab)] = pd.DataFrame({
        "t_h": cache.time_h, "predicted": external_m0[lab]
    })
    baseline_pred = np.interp(group["t_h"], cache.time_h, external_m0[lab])
    holdout_metric_rows.append(metric_row("M0_actual", lab, group, baseline_pred))

    refit_pred_grid = m0_grid_prediction(cache, m0_refit_parameters)
    candidate_curves[("M0_refit", lab)] = pd.DataFrame({
        "t_h": cache.time_h, "predicted": refit_pred_grid
    })
    holdout_metric_rows.append(
        metric_row("M0_refit", lab, group, np.interp(group["t_h"], cache.time_h, refit_pred_grid))
    )

    for model, values in effective_fits.items():
        predicted_full = values["gain"] * simulate_effective_filter(
            cache.time_h, external_qprod[lab], model, values["taus_h"]
        )
        candidate_curves[(model, lab)] = pd.DataFrame({
            "t_h": cache.time_h, "predicted": predicted_full
        })
        predicted_scored = np.interp(group["t_h"], cache.time_h, predicted_full)
        holdout_metric_rows.append(metric_row(model, lab, group, predicted_scored))

holdout_metrics_effective = pd.DataFrame(holdout_metric_rows)
display(holdout_metrics_effective[[
    "model", "lab", "RMSE_g_l_h", "NRMSE_peak", "onset_observed_h", "onset_predicted_h",
    "onset_error_h", "peak_time_observed_h", "peak_time_predicted_h", "AUC_relative_error"
]].round(4))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13.5, 10.2), sharex=True, layout="constrained")
for index, (ax, lab) in enumerate(zip(axes, EXTERNAL_LABS)):
    group = external_full_observed[external_full_observed["lab"].eq(lab)].sort_values("t_h")
    ax.plot(group["t_h"], group["co2_observed_g_l_h"], color="black", lw=1.7, label="Observado completo")
    for model, color, style in (
        ("M0_actual", "0.55", "--"),
        ("M0_refit", "#009E73", "-"),
        ("M1", "#0072B2", "-"),
        ("M2", "#D55E00", "-"),
    ):
        curve = candidate_curves[(model, lab)]
        width = 2.6 if model == SELECTED_MODEL else 1.7
        ax.plot(curve["t_h"], curve["predicted"], color=color, ls=style, lw=width, label=model)
    ax.set_title(lab, loc="left", fontsize=11)
    ax.set_ylabel(r"Flujo de CO$_2$ [g L$^{-1}$ h$^{-1}$]")
    ax.grid(alpha=0.25)
    if index == 0:
        ax.legend(loc="upper right", ncol=5, frameon=True, fontsize=8.0)
axes[-1].set_xlabel("t desde inoculación [h]")
fig.suptitle("Holdout externo LAB016–LAB018 · registro completo", fontsize=14)
display(fig)
plt.close(fig)

## Resumen del holdout externo

In [ ]:
holdout_summary = (
    holdout_metrics_effective.groupby("model", as_index=False)
    .agg(
        median_abs_onset_error_h=("onset_error_h", lambda values: float(np.nanmedian(np.abs(values)))),
        median_RMSE_g_l_h=("RMSE_g_l_h", "median"),
        median_abs_AUC_relative_error=("AUC_relative_error", lambda values: float(np.nanmedian(np.abs(values)))),
    )
)
peak_error = (
    holdout_metrics_effective.assign(
        abs_peak_time_error_h=lambda frame: np.abs(frame["peak_time_predicted_h"] - frame["peak_time_observed_h"])
    ).groupby("model")["abs_peak_time_error_h"].median()
)
holdout_summary["median_abs_peak_time_error_h"] = holdout_summary["model"].map(peak_error)
baseline_summary = holdout_summary[holdout_summary["model"].eq("M0_actual")].iloc[0]
holdout_summary["onset_reduction_vs_M0_pct"] = 100.0 * (
    1.0 - holdout_summary["median_abs_onset_error_h"] / baseline_summary["median_abs_onset_error_h"]
)
holdout_summary["RMSE_change_vs_M0_pct"] = 100.0 * (
    holdout_summary["median_RMSE_g_l_h"] / baseline_summary["median_RMSE_g_l_h"] - 1.0
)
bound_status = effective_parameters.groupby("model")["active_bound"].any().to_dict()
bound_status["M0_refit"] = bool(m0_refit_compare["active_bound"].any())
holdout_summary["timing_parameter_on_bound"] = [
    bool(bound_status.get(model, False)) for model in holdout_summary["model"]
]
display(holdout_summary.round(4))

selected_summary = holdout_summary[holdout_summary["model"].eq(SELECTED_MODEL)].iloc[0]
refit_summary = holdout_summary[holdout_summary["model"].eq("M0_refit")].iloc[0]
message = (
    f"El modelo elegido con LAB011–LAB012 fue **{SELECTED_MODEL}**. "
    f"En el holdout LAB016–LAB018 obtuvo un error mediano absoluto de onset de "
    f"**{selected_summary['median_abs_onset_error_h']:.1f} h**, frente a "
    f"**{baseline_summary['median_abs_onset_error_h']:.1f} h** de M0 y "
    f"**{refit_summary['median_abs_onset_error_h']:.1f} h** de M0 reajustado. "
    f"El RMSE mediano cambió **{selected_summary['RMSE_change_vs_M0_pct']:+.1f}%** "
    f"para {SELECTED_MODEL} y **{refit_summary['RMSE_change_vs_M0_pct']:+.1f}%** "
    f"para el M0 reajustado."
)
display(Markdown(message))

## Interpretación

- LAB019–LAB021 ya no son holdout: sus curvas limpias anteriores al segundo pulso participan en el ajuste con peso 2.
- LAB004 y LAB006 quedan completamente fuera del ajuste de M1/M2 y del reajuste de M0; LAB005, LAB007 y LAB008 conservan peso 1.
- LAB011–LAB012 se usan únicamente para elegir M1 o M2.
- LAB016–LAB018 son el único holdout externo. Sus condiciones iniciales se heredan de LAB013–LAB015 porque el Excel documenta que corresponden al mismo mosto; sus datos de CO2 no cambian ningún parámetro.
- El CO2 disuelto de LAB019–LAB021 sigue siendo una referencia diagnóstica y no entra en la función objetivo.
- La prueba modifica sólo capas de CO2: los filtros efectivos M1/M2 y el reajuste local de los parámetros de la capa de M0. `theta_full` y los artefactos anteriores permanecen congelados.
- M0 se recalibró localmente con los mismos lotes, tramos y pesos que M1/M2, reutilizando la maquinaria original `co2_cross.fit_matrix`; ese reajuste vive sólo en este notebook y `three_state_co2_parameters.csv` no cambia.
- Todas las figuras muestran las curvas completas.

In [ ]:

protected_after = {str(path): sha256(path) for path in PROTECTED_PATHS}
assert protected_before == protected_after, "un artefacto científico protegido cambió durante la prueba"
print("Auditoría de inmutabilidad: todos los artefactos anteriores conservan su SHA-256.")
print("Archivos de resultados o imágenes escritos por esta prueba: 0")
